In [1]:
from google.colab import files

uploaded = files.upload()

Saving lmdb_schema.json to lmdb_schema.json
Saving yago.json to yago.json
Saving drugbank_schema.json to drugbank_schema.json
Saving dbpedia_schema.json to dbpedia_schema.json


In [3]:
!pip install -q sentence-transformers

import json, re
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L12-v2"
FILES = {
    "lmdb":     "lmdb_schema.json",
    "yago":     "yago.json",
    "drugbank": "drugbank_schema.json",
    "dbpedia":  "dbpedia_schema.json",
}

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
print(f"Loaded {MODEL_NAME} on {device}")


PREFIX_RE = re.compile(r"^[A-Za-z0-9_]+/")

def clean_label(label):
    return PREFIX_RE.sub("", label)

def embed_items(items):
    cleaned = [clean_label(it["label"]) for it in items]
    embs = model.encode(cleaned, batch_size=128, show_progress_bar=False, convert_to_numpy=True)
    return [
        {"uri": it["uri"], "label": c, "original_label": it["label"], "embedding": e.tolist()}
        for it, c, e in zip(items, cleaned, embs)
    ]

for kg, path in FILES.items():
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    out = {
        "kg": kg,
        "model": MODEL_NAME,
        "dim": model.get_sentence_embedding_dimension(),
        "classes": embed_items(data["classes"]),
        "properties": embed_items(data["properties"]),
    }

    out_path = f"{kg}_embeddings.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(out, f)
    print(f"{kg}: {len(out['classes'])} classes, {len(out['properties'])} properties -> {out_path}")

print([p["label"] for p in json.load(open("drugbank_embeddings.json"))["properties"][:8]])

!zip -q kg_embeddings.zip lmdb_embeddings.json yago_embeddings.json drugbank_embeddings.json dbpedia_embeddings.json
from google.colab import files
files.download("kg_embeddings.zip")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded sentence-transformers/all-MiniLM-L12-v2 on cuda
lmdb: 49 classes, 69 properties -> lmdb_embeddings.json


/tmp/ipykernel_784/2847499674.py:41: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  "dim": model.get_sentence_embedding_dimension(),


yago: 388 classes, 141 properties -> yago_embeddings.json
drugbank: 4 classes, 118 properties -> drugbank_embeddings.json
dbpedia: 257 classes, 1276 properties -> dbpedia_embeddings.json
['label', 'equivalentClass', 'equivalentProperty', 'sameAs', 'absorption', 'affectedOrganism', 'ahfsCode', 'atcCode']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>